
**Grupo 11** · Hospital Regional del Altiplano · UCI como cola M/M/c y M/M/c/c.

Parámetros: $\lambda = 3.0+0.2((g-1)\bmod 5)=3.0$, $c=17+((g-1)\bmod 3)=18$, $\mu=0.25$, $a=\lambda/\mu=12$, $\rho=a/c=2/3$,
$k_{off}=\lceil c-a\rceil+3=9$, $T_b=7$ días.

In [1]:
import math, heapq, json, importlib.util, types, sys, os
from collections import deque
import numpy as np, pandas as pd
import matplotlib; matplotlib.use("Agg")
import matplotlib.pyplot as plt
from scipy import stats
pd.set_option("display.width", 200); pd.set_option("display.max_columns", 30)
os.makedirs("figs", exist_ok=True)

G = 11
LAM = 3.0 + 0.2*((G-1) % 5)
C   = 17 + ((G-1) % 3)
MU  = 0.25
A   = LAM/MU
RHO = A/C
TB  = 7.0
KOFF = math.ceil(C-A) + 3
print(dict(g=G, lam=LAM, c=C, mu=MU, a=A, rho=round(RHO,4), k_off=KOFF, Tb=TB))
R = {}   # resultados que luego se vuelcan al informe
R["params"] = dict(g=G, lam=LAM, c=C, mu=MU, a=A, rho=RHO, k_off=KOFF, Tb=TB)

{'g': 11, 'lam': 3.0, 'c': 18, 'mu': 0.25, 'a': 12.0, 'rho': 0.6667, 'k_off': 9, 'Tb': 7.0}


## Task 1.1 - Teoría
Fórmulas (diapositivas de la sesión de colas):

* Carga ofrecida $a=\lambda/\mu$, utilización $\rho=a/c$.
* **Erlang B** (recurrencia): $B(0,a)=1,\; B(k,a)=\dfrac{a\,B(k-1,a)}{k+a\,B(k-1,a)}$.
* **Erlang C**: $C(c,a)=\dfrac{B(c,a)}{1-\rho\,(1-B(c,a))}$ (probabilidad de esperar en M/M/c, $\rho<1$).
* $L_q=C(c,a)\dfrac{\rho}{1-\rho}$, $W_q=L_q/\lambda$, $W=W_q+1/\mu$, $L=L_q+a$.
* M/M/c/c: bloqueo $B(c,a)$, tasa efectiva $\lambda_{ef}=\lambda(1-B)$, camas ocupadas $L=\lambda_{ef}/\mu=a(1-B)$, $W=1/\mu$.
* Little: $L=\lambda_{(ef)}W$, $L_q=\lambda W_q$.

In [2]:
def erlang_b(c, a):
    # Recurrencia de Erlang B: B(k) = a B(k-1) / (k + a B(k-1))
    b = 1.0
    for k in range(1, c+1):
        b = a*b/(k + a*b)
    return b

def erlang_c(c, a):
    # Erlang C a partir de Erlang B: C = B / (1 - rho (1 - B))
    rho = a/c; b = erlang_b(c, a)
    return b/(1 - rho*(1 - b))

def teoria_mmc(lam, mu, c):
    a = lam/mu; rho = a/c
    Cw = erlang_c(c, a)
    Lq = Cw*rho/(1-rho); Wq = Lq/lam; W = Wq + 1/mu; L = Lq + a
    return dict(a=a, rho=rho, C=Cw, Lq=Lq, Wq_d=Wq, Wq_h=Wq*24, W_d=W, W_h=W*24, L=L)

def teoria_mmcc(lam, mu, c):
    a = lam/mu; B = erlang_b(c, a)
    lam_ef = lam*(1-B); L = lam_ef/mu
    return dict(a=a, B=B, lam_ef=lam_ef, L=L, util=L/c, W_d=1/mu, W_h=24/mu)

TW = teoria_mmc(LAM, MU, C); TS = teoria_mmcc(LAM, MU, C)
print("M/M/c   :", {k: round(v,5) for k,v in TW.items()})
print("M/M/c/c :", {k: round(v,5) for k,v in TS.items()})
# Little (teórico)
print("\nLittle M/M/c   : Lq - lam*Wq =", TW['Lq']-LAM*TW['Wq_d'], "| L - lam*W =", TW['L']-LAM*TW['W_d'])
print("Little M/M/c/c : L - lam_ef*W =", TS['L']-TS['lam_ef']*TS['W_d'])
print("Error si se usa lam en lugar de lam_ef: L=lam*W =", LAM*TS['W_d'], "vs", TS['L'],
      f"-> sobreestima {100*(LAM*TS['W_d']/TS['L']-1):.2f} %")
R["teoria_mmc"] = TW; R["teoria_mmcc"] = TS
R["little_lam_error_pct"] = 100*(LAM*TS['W_d']/TS['L']-1)

M/M/c   : {'a': 12.0, 'rho': 0.66667, 'C': 0.07561, 'Lq': 0.15123, 'Wq_d': 0.05041, 'Wq_h': 1.20984, 'W_d': 4.05041, 'W_h': 97.20984, 'L': 12.15123}
M/M/c/c : {'a': 12.0, 'B': 0.02654, 'lam_ef': 2.92037, 'L': 11.68148, 'util': 0.64897, 'W_d': 4.0, 'W_h': 96.0}

Little M/M/c   : Lq - lam*Wq = 0.0 | L - lam*W = 0.0
Little M/M/c/c : L - lam_ef*W = 0.0
Error si se usa lam en lugar de lam_ef: L=lam*W = 12.0 vs 11.681484585976623 -> sobreestima 2.73 %
